# Solar Power Production From Weather

Este notebook crea un flujo reproducible para entrenar un modelo que estime la energía generada por un sistema fotovoltaico a partir de variables meteorológicas (las mismas que obtenemos desde Open-Meteo).

## Flujo
1. Cargar y explorar `Solar Power Plant Data.csv`.
2. Limpiar datos, generar atributos cíclicos y normalizar la producción según la capacidad máxima registrada.
3. Entrenar un modelo basado en Gradient Boosting y evaluar MAE/RMSE sobre el último mes como conjunto de prueba.
4. Exponer una función utilitaria que reciba directamente la respuesta de Open-Meteo y devuelva la predicción de energía para cualquier capacidad de panel conocida.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.precision", 3)

In [4]:
DATASET_NAME = "Solar Power Plant Data.csv"
CANDIDATE_DIRS = [
    Path('datasets/clima paneles'),
    Path('notebooks/datasets/clima paneles'),
    Path('../datasets/clima paneles'),
    Path('../notebooks/datasets/clima paneles'),
]

for base in CANDIDATE_DIRS:
    candidate = base / DATASET_NAME
    if candidate.exists():
        DATA_PATH = candidate
        break
else:
    available = [str(base / DATASET_NAME) for base in CANDIDATE_DIRS]
    raise FileNotFoundError(f"No se encontró el dataset en ninguna de estas rutas: {available}")

print(f"Usando dataset: {DATA_PATH}")
df_raw = pd.read_csv(DATA_PATH)
print(f"Filas: {len(df_raw):,}")
df_raw.head()


Usando dataset: datasets/clima paneles/Solar Power Plant Data.csv
Filas: 8,760


,Date-Hour(NMT),WindSpeed,Sunshine,AirPressure,Radiation,AirTemperature,RelativeAirHumidity,SystemProduction
0,01.01.2017-00:00,0.6,0,1003.8,-7.4,0.1,97,0.0
1,01.01.2017-01:00,1.7,0,1003.5,-7.4,-0.2,98,0.0
2,01.01.2017-02:00,0.6,0,1003.4,-6.7,-1.2,99,0.0
3,01.01.2017-03:00,2.4,0,1003.3,-7.2,-1.3,99,0.0
4,01.01.2017-04:00,4.0,0,1003.1,-6.3,3.6,67,0.0


In [5]:
df = df_raw.copy()
df["timestamp"] = pd.to_datetime(df["Date-Hour(NMT)"], format="%d.%m.%Y-%H:%M")
numeric_cols = [
    "WindSpeed",
    "Sunshine",
    "AirPressure",
    "Radiation",
    "AirTemperature",
    "RelativeAirHumidity",
    "SystemProduction",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["SystemProduction"]).sort_values("timestamp").reset_index(drop=True)
df["Radiation"] = df["Radiation"].clip(lower=0)
df["Sunshine"] = df["Sunshine"].clip(lower=0, upper=100)

PLANT_CAPACITY_KW = df["SystemProduction"].max()
df["production_ratio"] = df["SystemProduction"] / PLANT_CAPACITY_KW

df["hour"] = df["timestamp"].dt.hour
df["month"] = df["timestamp"].dt.month
df["dayofyear"] = df["timestamp"].dt.dayofyear

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["year_day_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365)
df["year_day_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365)
df["sunshine_fraction"] = df["Sunshine"] / 100

print(f"Capacidad nominal estimada: {PLANT_CAPACITY_KW:.1f} kW")
df[["timestamp", "SystemProduction", "production_ratio"]].head()

Capacidad nominal estimada: 7701.0 kW


,timestamp,SystemProduction,production_ratio
0,2017-01-01 00:00:00,0.0,0.0
1,2017-01-01 01:00:00,0.0,0.0
2,2017-01-01 02:00:00,0.0,0.0
3,2017-01-01 03:00:00,0.0,0.0
4,2017-01-01 04:00:00,0.0,0.0


In [6]:
TARGET_COLUMN = "production_ratio"
FEATURE_COLUMNS = [
    "Radiation",
    "WindSpeed",
    "AirPressure",
    "AirTemperature",
    "RelativeAirHumidity",
    "sunshine_fraction",
    "hour_sin",
    "hour_cos",
    "year_day_sin",
    "year_day_cos",
]

df_model = df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN]).copy()
cutoff_date = df_model["timestamp"].max() - pd.Timedelta(days=30)
train_df = df_model[df_model["timestamp"] <= cutoff_date]
test_df = df_model[df_model["timestamp"] > cutoff_date]

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

print(f"Entrenamiento: {len(train_df):,} filas")
print(f"Prueba: {len(test_df):,} filas (desde {test_df['timestamp'].min().date()})")

Entrenamiento: 8,040 filas
Prueba: 720 filas (desde 2017-12-02)


In [7]:
model = GradientBoostingRegressor(
    random_state=42,
    n_estimators=600,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
)
model.fit(X_train, y_train)
print("Modelo entrenado")

Modelo entrenado


In [8]:
def regression_report(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred) * PLANT_CAPACITY_KW
    rmse = mean_squared_error(y_true, y_pred, squared=False) * PLANT_CAPACITY_KW
    r2 = r2_score(y_true, y_pred)
    return {"split": label, "MAE_kWh": mae, "RMSE_kWh": rmse, "R2": r2}

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)
metrics = pd.DataFrame([
    regression_report(y_train, train_pred, "train"),
    regression_report(y_test, test_pred, "test"),
]).set_index("split")
metrics

TypeError: got an unexpected keyword argument 'squared'

In [ ]:
preview_df = test_df[["timestamp", TARGET_COLUMN]].copy()
preview_df["prediction_ratio"] = test_pred
preview_df["prediction_kWh"] = preview_df["prediction_ratio"] * PLANT_CAPACITY_KW
preview_df["actual_kWh"] = preview_df[TARGET_COLUMN] * PLANT_CAPACITY_KW
preview_df.head(10)

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(test_df["timestamp"], test_df[TARGET_COLUMN] * PLANT_CAPACITY_KW, label="Real")
plt.plot(test_df["timestamp"], test_pred * PLANT_CAPACITY_KW, label="Modelo")
plt.title("Comparación último mes (kWh)")
plt.ylabel("kWh")
plt.xlabel("Fecha")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values()
importances.plot(kind="barh", figsize=(6, 4))
plt.title("Importancia de características")
plt.xlabel("Ganancia")
plt.tight_layout()
plt.show()

In [ ]:
feature_defaults = train_df[FEATURE_COLUMNS].median().to_dict()

def build_feature_vector_from_weather(weather_point, timestamp, defaults=feature_defaults):
    """Convierte la respuesta de Open-Meteo en el vector que espera el modelo.

    Parameters
    ----------
    weather_point : dict
        Debe contener al menos: temperature_2m, relative_humidity_2m, wind_speed_10m, shortwave_radiation,
        cloud_cover y opcionalmente surface_pressure.
    timestamp : str o pd.Timestamp
        Momento de la predicción (se usa para los atributos cíclicos).
    defaults : dict
        Valores medios para rellenar cualquier campo faltante.
    """
    ts = pd.to_datetime(timestamp)
    row = defaults.copy()
    row["Radiation"] = max(0.0, weather_point.get("shortwave_radiation", row["Radiation"]))
    row["WindSpeed"] = weather_point.get("wind_speed_10m", row["WindSpeed"])
    row["AirPressure"] = weather_point.get("surface_pressure", row["AirPressure"])
    row["AirTemperature"] = weather_point.get("temperature_2m", row["AirTemperature"])
    row["RelativeAirHumidity"] = weather_point.get("relative_humidity_2m", row["RelativeAirHumidity"])

    sunshine = weather_point.get("sunshine")
    if sunshine is None:
        cloud_cover = weather_point.get("cloud_cover")
        sunshine = 100 - cloud_cover if cloud_cover is not None else row["sunshine_fraction"] * 100
    sunshine = float(np.clip(sunshine, 0, 100))
    row["sunshine_fraction"] = sunshine / 100

    row["hour_sin"] = float(np.sin(2 * np.pi * ts.hour / 24))
    row["hour_cos"] = float(np.cos(2 * np.pi * ts.hour / 24))
    day_of_year = ts.timetuple().tm_yday
    row["year_day_sin"] = float(np.sin(2 * np.pi * day_of_year / 365))
    row["year_day_cos"] = float(np.cos(2 * np.pi * day_of_year / 365))

    return pd.DataFrame([row])[FEATURE_COLUMNS]

def predict_generation_from_weather(weather_point, timestamp, capacity_kw, model=model):
    features = build_feature_vector_from_weather(weather_point, timestamp)
    ratio = float(model.predict(features)[0])
    estimated_kwh = max(0.0, ratio * capacity_kw)
    return {
        "production_ratio": ratio,
        "estimated_energy_kwh": estimated_kwh,
        "capacity_kw": capacity_kw,
    }

sample_weather = {
    "temperature_2m": 26,
    "relative_humidity_2m": 48,
    "wind_speed_10m": 2.8,
    "shortwave_radiation": 650,
    "cloud_cover": 15,
    "surface_pressure": 1008,
}

predict_generation_from_weather(sample_weather, timestamp="2017-06-21 12:00", capacity_kw=5000)